# Lab 6 - Linear + Slack MCP Agent with Vaults

**Estimated time:** 30–40 min  ·  **Estimated cost:** ~$0.07


> **Networking note:** the Managed Agents harness reaches MCP servers over the public internet. Always point at a public, remote, or SaaS MCP endpoint like the Linear one used here. Local MCP tunnels are a research preview and are not used in this course.

![Architecture](assets/architecture.svg)

## Goal
Build an agent that connects to Linear and Slack through their public MCP servers. Both connections are authenticated **per end user** by credentials in the same vault. The agent triages Linear issues, files a new bug, then posts a completion update to Slack with the ticket title and identifier. You will learn the two-step MCP wiring (declare each server and toolset on the agent, supply auth on the session via one vault), and how MCP tool calls and auth errors surface in the event stream.

## Prerequisites
- A **Linear** account and a workspace where you can create issues.
- One **Claude Managed Agents vault containing both Linear and Slack MCP OAuth connections**. Paste its id into `LINEAR_VAULT_ID` in the setup cell.
- A Slack channel name in `SLACK_CHANNEL` where the agent may post the completion update.
- The notebook reads both MCP server URLs from those vault credentials; no manual MCP URL configuration is needed.
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ").strip()

print("Configure the Managed Agents vault containing Linear and Slack.")
existing_vault_id = os.environ.get("LINEAR_VAULT_ID", "").strip()
if existing_vault_id.startswith("sk-ant-") or (existing_vault_id and not existing_vault_id.startswith("vlt_")):
    print("Clearing invalid LINEAR_VAULT_ID. It must be a vault id starting with 'vlt_', not an API key.")
    os.environ.pop("LINEAR_VAULT_ID", None)

if not os.environ.get("LINEAR_VAULT_ID"):
    os.environ["LINEAR_VAULT_ID"] = input(
        "Linear + Slack vault ID from Claude Console (vlt_...): "
    ).strip()

if not os.environ.get("SLACK_CHANNEL"):
    os.environ["SLACK_CHANNEL"] = input(
        "Slack channel for completion updates (for example #research): "
    ).strip()

print("ANTHROPIC_API_KEY configured for this notebook session.")
print("LINEAR_VAULT_ID configured for this notebook session:", os.environ["LINEAR_VAULT_ID"])
print("SLACK_CHANNEL:", os.environ["SLACK_CHANNEL"])


In [ ]:
import os
from anthropic import Anthropic

# The previous cell configures ANTHROPIC_API_KEY and LINEAR_VAULT_ID for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
client = Anthropic()


def validate_vault_id(
    vault_id: str,
    env_var: str = "LINEAR_VAULT_ID",
    provider: str = "Linear",
) -> None:
    """Catch common copy/paste mistakes before sessions.create."""
    if not vault_id or vault_id.startswith("vlt_REPLACE"):
        raise RuntimeError(f"Set {env_var} to your Claude Managed Agents vault id.")
    if vault_id.startswith("sk-ant-"):
        raise RuntimeError(
            f"{env_var} currently contains an Anthropic API key. Paste the "
            f"{provider} vault id from Claude Console instead; it should start with 'vlt_'."
        )
    if not vault_id.startswith("vlt_"):
        raise RuntimeError(f"{env_var} should start with 'vlt_'. Got: {vault_id!r}")


def mcp_urls_from_vault(vault_id: str) -> dict[str, str]:
    """Read the Linear and Slack MCP URLs from one shared vault."""
    credentials = list(client.beta.vaults.credentials.list(vault_id, betas=BETAS))
    mcp_credentials = [
        credential for credential in credentials
        if getattr(getattr(credential, "auth", None), "type", None) == "mcp_oauth"
    ]
    urls: dict[str, str] = {}
    for provider in ("linear", "slack"):
        matches = [
            credential for credential in mcp_credentials
            if provider in (
                f"{getattr(credential, 'display_name', '') or ''} "
                f"{getattr(getattr(credential, 'auth', None), 'mcp_server_url', '')}"
            ).lower()
        ]
        if len(matches) != 1:
            names = [
                f"{getattr(credential, 'display_name', '') or credential.id}: "
                f"{getattr(getattr(credential, 'auth', None), 'mcp_server_url', '<no mcp url>')}"
                for credential in mcp_credentials
            ]
            raise RuntimeError(
                f"Expected exactly one {provider.title()} MCP OAuth credential "
                f"in vault {vault_id}; found {len(matches)}. "
                f"Available MCP credentials: {names or 'none'}"
            )
        urls[provider] = matches[0].auth.mcp_server_url
    return urls


LINEAR_VAULT_ID = os.environ.get("LINEAR_VAULT_ID", "").strip()
validate_vault_id(LINEAR_VAULT_ID, provider="Linear")
SLACK_CHANNEL = os.environ.get("SLACK_CHANNEL", "").strip()
if not SLACK_CHANNEL:
    raise RuntimeError("Set SLACK_CHANNEL to the channel for the completion update.")
MCP_URLS = mcp_urls_from_vault(LINEAR_VAULT_ID)

print("SDK ready, model:", MODEL)
print("LINEAR_VAULT_ID (Linear + Slack):", LINEAR_VAULT_ID)
print("Found MCP credentials:", sorted(MCP_URLS))


### Step 1 - Resolve the shared vault and declare both MCP servers on the agent
The existing vault supplies auth at session time. This notebook reads both MCP URLs from the Linear and Slack credentials in that vault, declares both servers **on the agent**, and adds both `mcp_toolset` entries to expose their tools to the model. Both toolsets use `always_allow` so the ticket creation and completion post run end to end.

In [ ]:
mcp_servers = [
    {"type": "url", "name": "linear", "url": MCP_URLS["linear"]},
    {"type": "url", "name": "slack", "url": MCP_URLS["slack"]},
]
tools = [
    {"type": "agent_toolset_20260401"},
    {"type": "mcp_toolset", "mcp_server_name": "linear",
     "default_config": {"permission_policy": {"type": "always_allow"}}},
    {"type": "mcp_toolset", "mcp_server_name": "slack",
     "default_config": {"permission_policy": {"type": "always_allow"}}},
]
vault_ids = [LINEAR_VAULT_ID]

agent = client.beta.agents.create(
    name="Linear Triage Agent",
    model=MODEL,
    system=(
        "You are a Linear assistant. You triage issues and file new ones "
        "via the linear MCP. When asked to file a bug, create a clear, "
        "well-titled issue with a concise description and reproduction "
        "steps. When asked to triage, list open or unassigned issues and "
        "suggest a priority for each. Always confirm the issue identifier "
        "(e.g. ENG-123) of anything you create or change. After creating "
        f"the Linear ticket, post a concise completion update to Slack channel "
        f"{SLACK_CHANNEL!r} with the ticket title and issue identifier. "
        "Do not report completion until both actions succeed."
    ),
    mcp_servers=mcp_servers,
    tools=tools,
    betas=BETAS,
)
print("agent.id =", agent.id)

### Step 2 - Create an environment and a session that references the vault
The environment uses **limited** networking but allows MCP servers (`allow_mcp_servers: True`), so the harness can reach both public endpoints. The session references the shared vault once via `vault_ids`; auth for both servers wires up server-side.

In [ ]:
env = client.beta.environments.create(
    name="linear-env",
    config={"type": "cloud", "networking": {"type": "limited",
            "allow_mcp_servers": True, "allowed_hosts": []}},
    betas=BETAS,
)
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=env.id,
    vault_ids=vault_ids,
    title="Linear triage + ticket + Slack update",
    betas=BETAS,
)
print("env.id =", env.id)
print("session.id =", session.id)

### Step 3 - Triage, create the issue, post to Slack, and stream
Send one turn and consume the stream. The agent must create the Linear issue before posting its completion update to Slack. Both tool calls surface as `agent.mcp_tool_use` events; MCP auth problems surface as non-fatal `session.error` events.

In [ ]:
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(session.id, events=[{
        "type": "user.message",
        "content": [{"type": "text",
            "text": ("Triage my unassigned Linear issues and tell me the top "
                     "three by likely priority. Then file a new bug titled "
                     "'Login crashes on empty password' with short reproduction "
                     f"steps. After creating it, post the ticket title and issue "
                     f"identifier to Slack channel {SLACK_CHANNEL!r}. Report the "
                     "new issue identifier and confirm the Slack post when done.")}],
    }], betas=BETAS)
    for event in stream:
        if event.type == "agent.message":
            for b in event.content:
                if b.type == "text":
                    print(b.text, end="", flush=True)
        elif event.type == "agent.mcp_tool_use":
            print(f"\n[mcp: {event.name}]")
        elif event.type == "session.error":
            print(f"\n[ERROR] {event.error}")
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
- `[mcp: linear_list_issues]` (or similar) as the agent reads your workspace.
- `[mcp: linear_create_issue]` as it files the new bug.
- `[mcp: slack_post_message]` (or similar) after the Linear issue is created.
- The agent's final turn names the new issue identifier (for example `ENG-142`).
- A new issue titled "Login crashes on empty password" visible in your Linear workspace.
- A completion message in `SLACK_CHANNEL` containing the ticket title and identifier.
- No `session.error` events about MCP auth.

## Troubleshooting
- **`invalid vault_id`** → `LINEAR_VAULT_ID` must be the vault id from Claude Console and should start with `vlt_`. Do not paste your `sk-ant-...` API key into this field.
- **`session.error: mcp_auth_failed` or "missing credential"** → one of the vault credentials is wrong or expired. Confirm `LINEAR_VAULT_ID` points to the shared vault containing valid Linear and Slack MCP OAuth credentials.
- **Expected exactly one Linear/Slack MCP OAuth credential** → the shared vault is missing that provider or contains duplicates. Fix the connections in Claude Console; do not configure MCP URLs manually.
- **No MCP tool calls appear** → the agent did not discover the server. Confirm all three pieces are wired: the `mcp_servers` array on the agent, the `mcp_toolset` entry in `tools`, and `vault_ids` passed at session creation.
- **OAuth scope errors when creating issues** → the Linear connection in the vault lacks write scope. Reconnect or rotate the Linear credential in Claude Console with issue write access.
- **Tool calls pause for confirmation** → confirm both `mcp_toolset` entries in Step 1 use `always_allow`. In production, use `always_ask` with an approval flow for write actions.
- **Networking blocked** → make sure the environment has `allow_mcp_servers: True`; with strict networking the harness cannot reach the public Linear endpoint.